# Batch sample review

Reviews a batch of finished (or in-progress) experiments together — the MERci equivalent of the old cluster-side review notebook, minus everything that needs MERlin to have already decoded the data (no barcode/cell-by-gene analysis here; see notebook 03/04 in this folder for mosaics/intensity plots of a single live experiment instead).

For each experiment: **checks whether its acquisition-time analysis already exists** (FOV stats/thumbnails/histograms, written during acquisition by `01_fov_scheduler.ipynb`) and **backfills anything missing** by calling the same per-file analysis MERci already uses, then reproduces the old notebook's per-round intensity/saturation comparison plots across the whole batch.

## 1 — Setup

In [ ]:
import os
import sys
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MERCI_DIR = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/analysis/)
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config       import ExperimentConfig
from MERci.common.metadata     import ExperimentMetadata
from MERci.common.experiment_info import load_experiment_info
from MERci.progress            import ProgressTracker
from MERci.progress_display    import ProgressReporter
from MERci.analysis.fov        import analyze_file, load_stats

## 2 — Select the batch

List the `experiment_info.yaml` path for each experiment to review together (written by `prepare_imaging/<variant>/05_create_experiment_info.ipynb`). Replaces the old notebook's master-CSV row filter (`experiment == '522' & imaged == 1`) with pointing directly at each sample's own info file.

In [ ]:
EXPERIMENT_INFO_PATHS = [
    # Path(r"D:\experiments\LT048_sample_26\metadata\experiment_info.yaml"),
    # Path(r"D:\experiments\LT048_sample_27\metadata\experiment_info.yaml"),
]

IMAGE_SUFFIX = ".zarr"   # must match what each experiment was acquired with

if not EXPERIMENT_INFO_PATHS:
    raise ValueError("Set EXPERIMENT_INFO_PATHS to at least one experiment_info.yaml path.")
print(f"{len(EXPERIMENT_INFO_PATHS)} experiment(s) selected.")

## 3 — Build config/metadata/tracker per experiment

In [ ]:
samples = []
for info_path in EXPERIMENT_INFO_PATHS:
    info_path  = Path(info_path)
    sample_dir = info_path.parent.parent   # metadata/experiment_info.yaml -> SAMPLE_DIR
    info       = load_experiment_info(info_path)

    config = ExperimentConfig(
        data_dir       = sample_dir / "data",
        metadata_dir   = sample_dir / "metadata",
        analysis_dir   = sample_dir / "analysis",
        settings_dir   = sample_dir / "settings",
        round_info_csv = sample_dir / "metadata" / "round_info.csv",
        positions_txt  = sample_dir / "positions" / f"positions_{info.sample_name}.txt",
        image_suffix   = IMAGE_SUFFIX,
    )
    meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                       config.data_dir, image_suffix=config.image_suffix)
    tracker = ProgressTracker(config.analysis_dir)

    samples.append({"info": info, "sample_dir": sample_dir, "config": config,
                    "meta": meta, "tracker": tracker})
    print(f"{info.sample_name:20s}  rounds={meta.n_rounds}  fovs={meta.n_fovs}  dir={sample_dir}")

## 4 — Verify analysis is available, backfill anything missing

If a sample's acquisition-time FOV scheduler already processed every file, this is a no-op (the common case, since analysis now runs *during* acquisition). Anything pending is analysed here directly, in parallel across a process pool (`config.resolved_n_workers` workers — the same convention `FOVScheduler` uses for the same `analyze_file` call).

In [ ]:
def backfill_pending(sample: dict) -> int:
    config, meta, tracker = sample["config"], sample["meta"], sample["tracker"]
    all_files = meta.all_expected_files()
    pending   = tracker.pending_fov_files(all_files)
    if not pending:
        return 0

    kwargs_common = dict(
        thumbnails_dir            = config.analysis_dir / "thumbnails",
        frame_width               = config.frame_width,
        frame_height              = config.frame_height,
        thumbnail_frames          = config.thumbnail_frames,
        thumbnail_size            = config.thumbnail_size,
        thumbnail_percentile_clip = config.thumbnail_percentile_clip,
        histogram_bins            = config.histogram_bins,
        histogram_range           = config.histogram_range,
    )

    reporter = ProgressReporter(total=len(pending), label=f"{sample['info'].sample_name}: backfilling")
    with ProcessPoolExecutor(max_workers=config.resolved_n_workers) as pool:
        futures = {
            pool.submit(
                analyze_file, fpath,
                stats_path     = tracker.stats_path(fpath),
                histogram_path = tracker.histogram_path(fpath),
                sentinel_path  = tracker.fov_sentinel(fpath),
                **kwargs_common,
            ): fpath
            for fpath in pending
        }
        for future in as_completed(futures):
            future.result()   # re-raise here if a worker errored on this FOV
            reporter.update()
    reporter.done()
    return len(pending)


for sample in samples:
    n_backfilled = backfill_pending(sample)
    smry = sample["tracker"].summary(sample["meta"])
    print(f"{sample['info'].sample_name:20s}  "
          f"backfilled={n_backfilled:4d}  "
          f"fovs_done={smry['files_fov_done']}/{smry['files_total']}  "
          f"rounds_done={smry['rounds_done']}/{smry['rounds_total']}")

## 5 — Load stats

Reads every completed stats CSV per sample into one combined DataFrame (adding a `sample_name` column), for the cross-sample plots below.

In [ ]:
def load_all_stats(sample: dict) -> pd.DataFrame:
    meta, tracker = sample["meta"], sample["tracker"]
    records = []
    for round_id in meta.valid_round_ids():
        round_obj = meta.rounds.get(round_id)
        if round_obj is None:
            continue
        for fov_id, file_list in round_obj.fov_files.items():
            for fpath in file_list:
                sp = tracker.stats_path(fpath)
                if not sp.exists():
                    continue
                df = load_stats(sp)
                df["round_id"] = round_id
                df["fov_id"]   = fov_id
                records.append(df)
    if not records:
        return pd.DataFrame()
    out = pd.concat(records, ignore_index=True)
    out["sample_name"] = sample["info"].sample_name
    return out


all_stats = pd.concat([load_all_stats(s) for s in samples], ignore_index=True)
print(f"Loaded {len(all_stats)} (sample x round x fov x frame) stat rows.")

## 6 — Per-round intensity / saturation comparison across the batch

One violin per sample, one figure per round — reproducing the old notebook's side-by-side sample comparison. `median` intensity and `max`/`p99` (as a saturation proxy — MERci's `measure_stats` does not compute an exact 'fraction of saturated pixels' metric) are plotted; add a dedicated `frac_saturated` stat to `analysis.fov.measure_stats` if exact parity with the old metric is needed.

In [ ]:
def plot_round_comparison(all_stats: pd.DataFrame, round_id: int, metric: str = "median") -> None:
    data = all_stats[all_stats["round_id"] == round_id]
    if data.empty:
        print(f"No data for round {round_id}.")
        return

    sample_names = sorted(data["sample_name"].unique())
    fig, ax = plt.subplots(figsize=(1.4 * len(sample_names) + 2, 4))
    values = [data.loc[data["sample_name"] == s, metric].dropna().values for s in sample_names]
    ax.violinplot(values, showmedians=True)
    ax.set_xticks(range(1, len(sample_names) + 1))
    ax.set_xticklabels(sample_names, rotation=30, ha="right")
    ax.set_ylabel(metric)
    ax.set_title(f"Round {round_id} — {metric} per frame, by sample")
    fig.tight_layout()
    plt.show()


if all_stats.empty:
    print("No stats to plot.")
else:
    for rid in sorted(all_stats["round_id"].unique()):
        plot_round_comparison(all_stats, rid, metric="median")
        plot_round_comparison(all_stats, rid, metric="max")